# 🎙️ IA gratis en Google Colab · Somos Simbióticos
**Clase «La IA a tu favor»** · [somossimbioticos.github.io](https://somossimbioticos.github.io)

Este cuaderno tiene 3 partes (puedes correr solo la que quieras):

1. **Clonar tu voz** con **Qwen3-TTS 1.7B** (código abierto, gratis).
2. **Voz profesional con la API de ElevenLabs** (necesitas tu propia llave).
3. **Crear imágenes** con un modelo abierto (SDXL-Turbo).

### Antes de empezar
* Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** (gratis).
* Corre las celdas en orden con ▶️ (o `Shift + Enter`).

> ⚠️ **Regla de oro:** clona SOLO tu voz o la de alguien que te dé permiso por escrito. Nunca para engañar a nadie.

In [ ]:
# 0) ¿Tengo GPU?
!nvidia-smi --query-gpu=name,memory.total --format=csv

---
## 1) 🗣️ Clonar tu voz con Qwen3-TTS 1.7B
Necesitas un audio tuyo de **5 a 15 segundos**, hablando claro y sin ruido (puede ser una nota de voz de WhatsApp) y **el texto exacto que dices en ese audio**.

In [ ]:
# 1.1 Instalar (tarda ~2 minutos)
!pip install -q -U qwen-tts soundfile

In [ ]:
# 1.2 Sube tu audio de referencia (mp3, wav, ogg, m4a…)
from google.colab import files
subido = files.upload()
REF_AUDIO = list(subido.keys())[0]
print("Audio:", REF_AUDIO)

In [ ]:
# 1.3 Escribe EXACTAMENTE lo que dices en ese audio (mejora mucho la calidad)
REF_TEXTO = "Hola, soy yo hablando para probar cómo suena mi voz clonada con inteligencia artificial."  # @param {type:"string"}

# Si no sabes el texto exacto, pon SOLO_TIMBRE = True (funciona, pero con menos parecido)
SOLO_TIMBRE = False  # @param {type:"boolean"}

In [ ]:
# 1.4 Cargar el modelo (se descarga ~4 GB la primera vez)
import torch
from qwen_tts import Qwen3TTSModel

modelo = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.float16,           # la GPU T4 de Colab funciona mejor en float16
    attn_implementation="sdpa",    # flash-attention no está disponible en la T4
)
print("Modelo listo ✅")

In [ ]:
# 1.5 ¡Haz que tu voz diga lo que quieras!
import soundfile as sf
from IPython.display import Audio, display

TEXTO = "Bienvenidos a la clase de inteligencia artificial de Somos Simbióticos. Hoy vamos a aprender a usar la IA a nuestro favor."  # @param {type:"string"}

wavs, sr = modelo.generate_voice_clone(
    text=TEXTO,
    language="Spanish",
    ref_audio=REF_AUDIO,
    ref_text=None if SOLO_TIMBRE else REF_TEXTO,
    x_vector_only_mode=SOLO_TIMBRE,
)
sf.write("mi_voz_clonada.wav", wavs[0], sr)
display(Audio("mi_voz_clonada.wav"))

In [ ]:
# 1.6 Descargar el audio
files.download("mi_voz_clonada.wav")

---
## 2) 🎧 Voz profesional con la API de ElevenLabs
1. Crea tu cuenta en [elevenlabs.io](https://elevenlabs.io) y copia tu llave en **Perfil → API Keys**.
2. En Colab, toca la 🔑 **llave de la izquierda (Secretos)** → *Agregar nuevo secreto* → nombre `ELEVENLABS_API_KEY`, pega tu llave y activa el acceso del cuaderno.

> 🔒 **Nunca pegues tu llave dentro del código** ni la compartas en capturas o presentaciones: cualquiera podría gastar tu saldo.

In [ ]:
# 2.1 Leer la llave desde los Secretos de Colab y ver las voces disponibles
import requests
from google.colab import userdata

LLAVE = userdata.get("ELEVENLABS_API_KEY")
r = requests.get("https://api.elevenlabs.io/v1/voices", headers={"xi-api-key": LLAVE})
r.raise_for_status()
voces = r.json()["voices"]
for v in voces[:15]:
    print(f'{v["name"]:<20} {v["voice_id"]}  {v.get("labels", {}).get("gender", "")}')

In [ ]:
# 2.2 Convertir texto a voz (modelo multilingüe: habla muy bien español)
VOZ_ID = voces[0]["voice_id"]  # cámbialo por el voice_id que más te guste
TEXTO_11 = "Hola, esta voz fue creada con la API de ElevenLabs desde Google Colab."  # @param {type:"string"}

r = requests.post(
    f"https://api.elevenlabs.io/v1/text-to-speech/{VOZ_ID}",
    headers={"xi-api-key": LLAVE, "Content-Type": "application/json"},
    json={"text": TEXTO_11, "model_id": "eleven_multilingual_v2"},
)
r.raise_for_status()
open("voz_elevenlabs.mp3", "wb").write(r.content)
display(Audio("voz_elevenlabs.mp3"))

---
## 3) 🎨 Crear imágenes gratis con la GPU de Colab (SDXL-Turbo)
Genera una imagen en ~2 segundos. *Licencia de SDXL-Turbo: uso personal y de investigación (no comercial).*

In [ ]:
# 3.1 Instalar y cargar (~1 minuto)
!pip install -q -U diffusers transformers accelerate
import torch
from diffusers import AutoPipelineForText2Image

imagenes = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")
print("Listo ✅")

In [ ]:
# 3.2 Describe tu imagen (en inglés suele salir mejor)
PROMPT = "a hummingbird drinking from a colorful flower, macro photography, golden hour, highly detailed"  # @param {type:"string"}

img = imagenes(prompt=PROMPT, num_inference_steps=1, guidance_scale=0.0, height=512, width=512).images[0]
img.save("mi_imagen.png")
img

---
### ¿Quieres todo esto sin programar?
En **[Simbi](https://somossimbioticos.github.io)** tienes voz, música, imágenes, subtítulos, doblaje y más con un clic.
Código de prueba de la clase: **SOMOS2026**.